# SelectiveNet — California Housing (MLP)

In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, json, os, sys
from sklearn.metrics import r2_score
sys.path.insert(0, os.path.join('.', '..'))
from shared.data_utils import load_california_housing, build_mlp_encoder

CONFIG = {'method': 'selectivenet', 'hidden_dims': [128, 64], 'target_coverage': 0.7,
          'lambda_cov': 10.0, 'epochs': 200, 'batch_size': 32, 'lr': 1e-3, 'seeds': [42, 43, 44]}
RESULT_DIR = os.path.join('.', 'results', 'selectivenet')
os.makedirs(RESULT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
class SelectiveNet(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        self.encoder, h_dim = build_mlp_encoder(input_dim, hidden_dims)
        self.pred_head = nn.Linear(h_dim, 1)
        self.sel_head = nn.Linear(h_dim, 1)
    def forward(self, x):
        h = self.encoder(x)
        return self.pred_head(h), torch.sigmoid(self.sel_head(h))

def selective_loss(y, y_hat, s, target_cov, lam):
    mse = (y - y_hat) ** 2
    cov_loss = lam * F.relu(target_cov - s.mean()) ** 2
    reg_loss = (s * mse).sum() / (s.sum() + 1e-8)
    return reg_loss + cov_loss
print('Model defined.')

In [ ]:
def train_one_seed(seed):
    print(f'\n--- Seed {seed} ---')
    X_train, y_train, X_val, y_val, X_test, y_test, scaler, input_dim = \
        load_california_housing(random_state=seed)
    train_ds = torch.utils.data.TensorDataset(X_train, y_train)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
    model = SelectiveNet(input_dim, CONFIG['hidden_dims']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    best_state, best_val = None, float('inf')
    for epoch in range(CONFIG['epochs']):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            y_hat, s = model(xb)
            loss = selective_loss(yb, y_hat, s, CONFIG['target_coverage'], CONFIG['lambda_cov'])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            y_hat_v, s_v = model(X_val.to(DEVICE))
            vl = selective_loss(y_val.to(DEVICE), y_hat_v, s_v, CONFIG['target_coverage'], CONFIG['lambda_cov']).item()
        if vl < best_val: best_val = vl; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        y_hat, s = model(X_test.to(DEVICE))
        y_pred = y_hat.cpu().numpy().squeeze()
        scores = s.cpu().numpy().squeeze()
        y_true = y_test.numpy().squeeze()
    return y_pred, scores, y_true

for seed in CONFIG['seeds']:
    y_pred, scores, y_true = train_one_seed(seed)
    sd = os.path.join(RESULT_DIR, f'seed_{seed}'); os.makedirs(sd, exist_ok=True)
    np.save(os.path.join(sd, 'test_predictions.npy'), y_pred)
    np.save(os.path.join(sd, 'test_scores.npy'), scores)
    np.save(os.path.join(sd, 'test_labels.npy'), y_true)
    print(f'  R²: {r2_score(y_true, y_pred):.4f}')
print(f'\nDone. Saved to {RESULT_DIR}')